## 1. Verify PyTorch Installation

**Why this cell:** Before writing any code, we confirm (a) PyTorch is installed in this
notebook's interpreter, and (b) whether a GPU is available. Everything else in this
notebook depends on this check.

**Line by line:**
- `import torch` — loads the PyTorch library.
- `print(torch.__version__)` — prints the installed version. The suffix tells the build:
  `2.10.0+cu128` = compiled with the CUDA 12.8 runtime. Even a CUDA build runs fine on CPU.
- `torch.cuda.is_available()` — returns `True` only if a working NVIDIA GPU **and** its
  drivers are present. `False` here is expected on this machine (no GPU), so the whole
  notebook runs on CPU.

**Teacher cross-questions:**
- *Q: Why check the device before anything else?*
  A: Every model and tensor must live on the **same device** (CPU or GPU) for any
  computation. Mixing them raises `RuntimeError: Expected all tensors to be on the
  same device`. So we decide the device once, first, and reuse it everywhere.
- *Q: If a GPU is available, are we forced to use it?*
  A: No — but a GPU executes matrix operations in parallel, so training is far faster.
  For this demo we only do inference on a few images, so CPU is perfectly fine.

In [10]:
import torch
print(torch.__version__)
print("GPU Available:", torch.cuda.is_available())

2.10.0+cu128
GPU Available: False


In [14]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [15]:
import numpy as np

# 3 data points, har ek mein 2 features (jaise X), 1 label (y)
X_np = np.array([[1, 2], [3, 4], [5, 6]])
y_np = np.array([[1], [2], [3]])

X = torch.from_numpy(X_np).float().to(device)
y = torch.from_numpy(y_np).float().to(device)

print(X.shape)  # torch.Size([3, 2])
print(y.shape)  # torch.Size([3, 1])

torch.Size([3, 2])
torch.Size([3, 1])


In [16]:
import torch.nn as nn

class Model_0(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=2, out_features=5)
        self.layer_2 = nn.Linear(in_features=5, out_features=1)

    def forward(self, x):
        return self.layer_2(self.layer_1(x))

model_0 = Model_0().to(device)
print(model_0)

Model_0(
  (layer_1): Linear(in_features=2, out_features=5, bias=True)
  (layer_2): Linear(in_features=5, out_features=1, bias=True)
)


In [17]:
class Model_1(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layer_stack = nn.Sequential(
            nn.Linear(in_features=2, out_features=5),
            nn.ReLU(),                      # activation function
            nn.Linear(in_features=5, out_features=1)
        )

    def forward(self, x):
        return self.linear_layer_stack(x)

model_1 = Model_1().to(device)
print(model_1)

Model_1(
  (linear_layer_stack): Sequential(
    (0): Linear(in_features=2, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=1, bias=True)
  )
)


In [18]:
# Loss: model ne kitni galat prediction ki (regression ke liye MSE)
criterion = nn.MSELoss()

# Optimizer: weights ko kaise update karna hai (Adam sabse common)
optimizer = torch.optim.Adam(model_1.parameters(), lr=0.001)

print("Loss: MSE | Optimizer: Adam")

Loss: MSE | Optimizer: Adam


In [19]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, y)          # X aur y ko saath jodo
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

for batch_features, batch_labels in dataloader:
    print("Batch X shape:", batch_features.shape, "| Batch y shape:", batch_labels.shape)

Batch X shape: torch.Size([3, 2]) | Batch y shape: torch.Size([3, 1])
